# 🧬 Day 03a: Prototype Pattern — Clone Instead of Create

---

## 🎯 What You'll Master Today
- Prototype: clone existing objects instead of building from scratch
- Shallow vs deep copy — critical difference
- Python's `copy` module and `__copy__`/`__deepcopy__`
- When cloning beats constructing

---

## 🎭 The Analogy

Think of **photocopying a document**:
- You have a perfect original document (prototype)
- Instead of typing it from scratch, you photocopy it and make small changes
- The copy is independent — marking up the copy doesn't affect the original

---

In [ ]:
# ============================================
# 1. The Problem — Expensive Object Creation
# ============================================
import copy
import time

class GameCharacter:
    """Imagine this takes 2 seconds to load assets."""
    
    def __init__(self, name: str, level: int, skills: list, inventory: dict):
        # Simulating expensive initialization
        self.name = name
        self.level = level
        self.skills = skills
        self.inventory = inventory
    
    def clone(self) -> 'GameCharacter':
        """Prototype method — deep clone."""
        return copy.deepcopy(self)
    
    def __str__(self):
        return f"{self.name} (Lv{self.level}) Skills: {self.skills}"


# Create a template character
warrior_template = GameCharacter(
    name="Warrior",
    level=1,
    skills=["slash", "block"],
    inventory={"sword": 1, "shield": 1, "potion": 3}
)

# Clone and customize instead of creating from scratch
player1 = warrior_template.clone()
player1.name = "Sarthak"
player1.skills.append("power_strike")  # doesn't affect template!

player2 = warrior_template.clone()
player2.name = "Alice"
player2.level = 5

print(f"Template: {warrior_template}")
print(f"Player1:  {player1}")
print(f"Player2:  {player2}")
print(f"\nTemplate unchanged? Skills = {warrior_template.skills}")  # ✅ unchanged

In [ ]:
# ============================================
# 2. Shallow vs Deep Copy — CRITICAL Difference
# ============================================
import copy

original = {"name": "Sarthak", "scores": [90, 85, 95], "meta": {"role": "admin"}}

# Shallow copy — inner objects are SHARED
shallow = copy.copy(original)
shallow["name"] = "Alice"          # ✅ separate — strings are immutable
shallow["scores"].append(100)       # ❌ affects original! (shared list)

print("After shallow copy mutation:")
print(f"  Original scores: {original['scores']}")  # [90, 85, 95, 100] — mutated!
print(f"  Shallow scores:  {shallow['scores']}")    # [90, 85, 95, 100]

# Deep copy — everything is independent
original2 = {"name": "Sarthak", "scores": [90, 85, 95], "meta": {"role": "admin"}}
deep = copy.deepcopy(original2)
deep["scores"].append(100)          # ✅ doesn't affect original
deep["meta"]["role"] = "user"       # ✅ doesn't affect original

print("\nAfter deep copy mutation:")
print(f"  Original scores: {original2['scores']}")  # [90, 85, 95] — safe!
print(f"  Deep scores:     {deep['scores']}")        # [90, 85, 95, 100]
print(f"  Original role:   {original2['meta']['role']}")  # admin — safe!

```
╔══════════════════╦══════════════════════════════════════════════╗
║  Copy Type        ║  Behavior                                   ║
╠══════════════════╬══════════════════════════════════════════════╣
║  Assignment (=)   ║  Same object, different name (alias)       ║
║  Shallow copy     ║  New outer object, inner objects SHARED    ║
║  Deep copy        ║  New everything — fully independent        ║
╚══════════════════╩══════════════════════════════════════════════╝
```

> **Interview Tip:** "I use shallow copy for flat objects. Deep copy when the object contains nested mutables (lists, dicts, other objects)."

---

In [ ]:
# ============================================
# 3. Prototype Registry — Template Store
# ============================================
import copy
from abc import ABC, abstractmethod

class Prototype(ABC):
    @abstractmethod
    def clone(self): pass


class Document(Prototype):
    def __init__(self, title: str, content: str, formatting: dict):
        self.title = title
        self.content = content
        self.formatting = formatting
    
    def clone(self) -> 'Document':
        return copy.deepcopy(self)
    
    def __str__(self):
        return f"📄 {self.title} | Font: {self.formatting.get('font')}"


class PrototypeRegistry:
    """Store prototypes, clone when needed."""
    
    def __init__(self):
        self._prototypes: dict[str, Prototype] = {}
    
    def register(self, name: str, prototype: Prototype):
        self._prototypes[name] = prototype
    
    def create(self, name: str) -> Prototype:
        proto = self._prototypes.get(name)
        if not proto:
            raise KeyError(f"No prototype registered: {name}")
        return proto.clone()


# Register templates
registry = PrototypeRegistry()
registry.register("report", Document(
    "Quarterly Report", "", {"font": "Arial", "size": 12, "margins": "1in"}
))
registry.register("memo", Document(
    "Internal Memo", "", {"font": "Calibri", "size": 11, "margins": "0.5in"}
))

# Clone and customize
doc1 = registry.create("report")
doc1.title = "Q4 2024 Sales Report"
doc1.content = "Revenue increased by 15%..."

doc2 = registry.create("memo")
doc2.title = "Team Standup Notes"

print(doc1)
print(doc2)

## 📝 Interview Questions — Practice These

| # | Question | Key Point |
|---|----------|------------|
| 1 | What is the Prototype pattern? | Clone existing objects instead of creating new ones from scratch |
| 2 | Shallow vs deep copy? | Shallow shares inner objects. Deep copies everything. Use deep for nested mutables |
| 3 | When to use Prototype? | Expensive object creation, template-based creation, avoiding subclass explosion |
| 4 | How does Python implement it? | copy.copy() for shallow, copy.deepcopy() for deep. Override __copy__/__deepcopy__ for custom |
| 5 | Prototype vs Factory? | Factory creates from scratch. Prototype clones existing. Use prototype when init is expensive |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Prototype = clone instead of construct               │
## │  • Deep copy for nested objects (use copy.deepcopy)      │
## │  • Shallow copy shares inner mutables — dangerous!       │
## │  • Registry pattern stores named prototypes              │
## │  • Use when creation is expensive or template-based      │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Adapter & Decorator** — making incompatible things work together and adding behavior